# **Supuestos (A1)-(A5) de van den Driessche y Watmough con `check_assumptions()`**

El método de la matriz de próxima generación escribe cada compartimento como
$$\dot x_i = \mathcal{F}_i(x) - \mathcal{V}_i(x), \qquad \mathcal{V}_i = \mathcal{V}_i^- - \mathcal{V}_i^+,$$
donde $\mathcal{F}_i$ son las nuevas infecciones, $\mathcal{V}_i^+$ las demás entradas y $\mathcal{V}_i^-$ las salidas. El Teorema 2 de van den Driessche y Watmough (2002) garantiza que **$\mathcal{R}_0 = \rho(FV^{-1})$ es un umbral** (el DFE es localmente asintóticamente estable si $\mathcal{R}_0 < 1$ e inestable si $\mathcal{R}_0 > 1$) solo si se cumplen cinco supuestos:

| | Enunciado |
|:--:|---|
| (A1) | Si $x \ge 0$, entonces $\mathcal{F}_i,\ \mathcal{V}_i^+,\ \mathcal{V}_i^- \ge 0$ |
| (A2) | Si $x_i = 0$, entonces $\mathcal{V}_i^- = 0$ |
| (A3) | $\mathcal{F}_i = 0$ para los compartimentos no infectados ($i > m$) |
| (A4) | Si $x \in X_s$ (todos los infectados en cero), entonces $\mathcal{F}_i = 0$ y $\mathcal{V}_i^+ = 0$ para $i \le m$ |
| (A5) | Con $\mathcal{F} = 0$, los valores propios de $Df(x_0)$ tienen parte real negativa: los de $V$ y los de $J_4$ tienen parte real positiva |

`model.check_assumptions()` decide cada supuesto como **cumple**, **no cumple** o **no se pudo decidir**, e indica el fundamento:

* **por construcción**: lo garantiza la forma en que pyR0compute arma el modelo, como (A3);
* **demostración simbólica**: vale para todo valor positivo de los parámetros;
* **en los valores dados**: decidido en el punto `values={...}`;
* **muestras aleatorias**: si la violación aparece en todas las muestras de parámetros, *no cumple*; si el supuesto se cumple en todas las muestras pero no se pudo demostrar, o se cumple solo en algunas, *no se pudo decidir* (con `values=` se decide en un punto).

El informe se entrega en LaTeX, listo para un artículo (`to_latex()`) o en Markdown para el cuaderno.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/05_supuestos_vdw.ipynb)

In [1]:
# Instala pyR0compute desde PyPI con el extra [global] (SciPy >= 1.11).
# La versión en desarrollo: %pip install "pyR0compute[global] @ git+https://github.com/Yvan-BM-Git/pyR0compute"
%pip install "pyR0compute[global]>=0.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings                               # Algunos modelos tienen varios DFE (aviso)
import numpy as np                            # Cálculo numérico
import sympy as sp                            # Cálculo simbólico
from pyr0compute import R0Model               # Clase principal de la librería
from IPython.display import Markdown, display # Para mostrar el informe


def seccion(informe, clave):
    '''Muestra solo la sección de un supuesto (por ejemplo "A4") del informe en Markdown.'''
    texto = informe.to_latex("markdown")
    inicio = texto.find(f"### ({clave})")
    fin = texto.find("\n### ", inicio + 1)
    display(Markdown(texto[inicio:fin]))

## 1. Un modelo que cumple los cinco supuestos

El modelo con linfocitos T helper de Cuesta-Herrera et al. (2025), *Math. Biosci. Eng.* 22(11):2807-2825. Los cinco supuestos se demuestran para todo valor positivo de los parámetros. En (A5), $J_4$ tiene valores propios $d$ y $c$, los mismos que aparecen en la demostración del Teorema 1 del artículo.

Al evaluar el informe en una celda, Jupyter lo muestra completo: tabla resumen, descomposición $\mathcal{F}_i$, $\mathcal{V}_i^+$, $\mathcal{V}_i^-$, el detalle de cada supuesto, las consecuencias del Lema 1 ($F \ge 0$ y $V$ una M-matriz no singular) y la conclusión.

In [3]:
modelo_th = R0Model('''
    dE/dt  = lambda - d*E - kappa*E*V
    dI/dt  = kappa*E*V - alpha*I - beta*Th*I
    dV/dt  = nu*I - mu*V - tau*Th*V
    dTh/dt = b - c*Th + gamma*I*Th
''', infected=["I", "V"])

informe_th = modelo_th.check_assumptions(language="es")
informe_th

## Supuestos de van den Driessche y Watmough (2002)

Compartimentos infectados ($i \le m$): $I, V$. Sin valores de los parámetros: cada supuesto se demuestra para todo parámetro positivo cuando es posible; si no, se explora con muestras aleatorias.

| Supuesto | Enunciado | Resultado | Fundamento |
|:--:|---|:--:|---|
| (A1) | flujos no negativos | **cumple** | demostración simbólica para todo parámetro positivo |
| (A2) | sin salidas de un compartimento vacío | **cumple** | demostración simbólica para todo parámetro positivo |
| (A3) | sin nuevas infecciones en no infectados | **cumple** | por construcción |
| (A4) | el subespacio libre de infección es invariante | **cumple** | demostración simbólica para todo parámetro positivo |
| (A5) | el DFE es estable sin nuevas infecciones | **cumple** | demostración simbólica para todo parámetro positivo |

### Descomposición utilizada

$\dot x_i = \mathcal{F}_i - \mathcal{V}_i$, con $\mathcal{V}_i = \mathcal{V}_i^- - \mathcal{V}_i^+$ ($\mathcal{V}_i^+$: otras entradas; $\mathcal{V}_i^-$: salidas).

$$
\begin{aligned}
\mathcal{F}_{E} &= 0, & \mathcal{V}_{E}^+ &= \lambda, & \mathcal{V}_{E}^- &= E V \kappa + E d \\
\mathcal{F}_{I} &= E V \kappa, & \mathcal{V}_{I}^+ &= 0, & \mathcal{V}_{I}^- &= I Th \beta + I \alpha \\
\mathcal{F}_{V} &= 0, & \mathcal{V}_{V}^+ &= I \nu, & \mathcal{V}_{V}^- &= Th V \tau + V \mu \\
\mathcal{F}_{Th} &= 0, & \mathcal{V}_{Th}^+ &= I Th \gamma + b, & \mathcal{V}_{Th}^- &= Th c
\end{aligned}
$$

### (A1) flujos no negativos

Si $x \ge 0$, entonces $\mathcal{F}_i,\ \mathcal{V}_i^+,\ \mathcal{V}_i^- \ge 0$ para $i = 1,\dots,n$.

Resultado: **cumple** (demostración simbólica para todo parámetro positivo).

* $\mathcal{V}_i^+$ y $\mathcal{V}_i^-$ reúnen los términos negativos y positivos de $\mathcal{V}_i$; son no negativos cuando cada término tiene signo definido.

### (A2) sin salidas de un compartimento vacío

Si $x_i = 0$, entonces $\mathcal{V}_i^- = 0$.

Resultado: **cumple** (demostración simbólica para todo parámetro positivo).

* $\left.\mathcal{V}_{E}^-\right\rvert_{E = 0} = 0$: **cumple**
* $\left.\mathcal{V}_{I}^-\right\rvert_{I = 0} = 0$: **cumple**
* $\left.\mathcal{V}_{V}^-\right\rvert_{V = 0} = 0$: **cumple**
* $\left.\mathcal{V}_{Th}^-\right\rvert_{Th = 0} = 0$: **cumple**

### (A3) sin nuevas infecciones en no infectados

$\mathcal{F}_i = 0$ para $i > m$ (compartimentos no infectados).

Resultado: **cumple** (por construcción).

* pyR0compute define $\mathcal{F}_i$ solo para los compartimentos infectados y rechaza términos de nuevas infecciones en los no infectados.

### (A4) el subespacio libre de infección es invariante

Si $x \in X_s$, entonces $\mathcal{F}_i(x) = 0$ y $\mathcal{V}_i^+(x) = 0$ para $i = 1,\dots,m$.

Resultado: **cumple** (demostración simbólica para todo parámetro positivo).

* $\left.\mathcal{F}_{I}\right\rvert_{X_s} = 0$: **cumple**
* $\left.\mathcal{V}_{I}^+\right\rvert_{X_s} = 0$: **cumple**
* $\left.\mathcal{F}_{V}\right\rvert_{X_s} = 0$: **cumple**
* $\left.\mathcal{V}_{V}^+\right\rvert_{X_s} = 0$: **cumple**
* $X_s = \{x \ge 0 : x_i = 0,\ i \le m\}$; los no infectados quedan libres, no fijos en $x_0$.

### (A5) el DFE es estable sin nuevas infecciones

Con $\mathcal{F} = 0$, todos los valores propios de $Df(x_0)$ tienen parte real negativa; equivalentemente, los de $V$ y los de $J_4$ tienen parte real positiva.

Resultado: **cumple** (demostración simbólica para todo parámetro positivo).

* $V = \left[\begin{matrix}\frac{\alpha c + b \beta}{c} & 0\\- \nu & \frac{b \tau + c \mu}{c}\end{matrix}\right]$: **cumple**
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}d & 0\\0 & c\end{matrix}\right]$: **cumple**
* $V$: $\alpha + \frac{b \beta}{c} > 0$, $\frac{b \tau}{c} + \mu > 0$
* $J_4$: $d > 0$, $c > 0$

### Consecuencias (Lema 1)

* $F \ge 0$ (por entradas): **cumple**
* $V$ tiene patrón de signos Z ($V_{ij} \le 0$, $i \ne j$): **cumple**
* $\partial \mathcal{V}_i / \partial x_j (x_0) = 0$ para $i \le m < j$: **cumple**

Si $V$ tiene patrón Z y sus valores propios tienen parte real positiva, $V$ es una M-matriz no singular y $V^{-1} \ge 0$.

### Conclusión

Se cumplen (A1) a (A5). Por el Teorema 2, el DFE $x_0$ es localmente asintóticamente estable si $\mathcal{R}_0 < 1$ e inestable si $\mathcal{R}_0 > 1$.

También hay un resumen en texto y acceso a cada resultado:

In [4]:
print(informe_th)
print()
print("¿Se aplica el Teorema 2?", informe_th.holds)
print("Estado de cada supuesto:", informe_th.status)
print("(A5):", informe_th["A5"].status, "|", informe_th["A5"].basis)

van den Driessche & Watmough (2002) assumptions:
  (A1) holds     [symbolic proof for all positive parameters]  non-negative flows
  (A2) holds     [symbolic proof for all positive parameters]  no outflow from an empty compartment
  (A3) holds     [by construction]  no new infections in uninfected compartments
  (A4) holds     [symbolic proof for all positive parameters]  the disease-free subspace is invariant
  (A5) holds     [symbolic proof for all positive parameters]  the DFE is stable without new infections
Theorem 2 applies (R0 is a threshold).

¿Se aplica el Teorema 2? True
Estado de cada supuesto: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}
(A5): holds | symbolic


### El informe en LaTeX para un artículo

`to_latex()` entrega el informe en LaTeX (requiere `amsmath`); con `standalone=True` es un archivo que compila tal cual; aquí se guarda en `outputs/05_supuestos_vdw.tex`. Se puede escribir en inglés con `language="en"`.

In [5]:
tex = informe_th.to_latex(standalone=True)
import os
os.makedirs("outputs", exist_ok=True)
with open("outputs/05_supuestos_vdw.tex", "w") as f:
    f.write(tex)
print(tex[:1400], "...")

\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage{amsmath,amssymb}
\usepackage[margin=2cm]{geometry}
\begin{document}

\section*{Supuestos de van den Driessche y Watmough (2002)}

Compartimentos infectados ($i \le m$): $I, V$. Sin valores de los parámetros: cada supuesto se demuestra para todo parámetro positivo cuando es posible; si no, se explora con muestras aleatorias.

\begin{center}
\begin{tabular}{c p{0.34\textwidth} c p{0.3\textwidth}}
\hline
Supuesto & Enunciado & Resultado & Fundamento \\
\hline
(A1) & flujos no negativos & \textbf{cumple} & demostración simbólica para todo parámetro positivo \\
(A2) & sin salidas de un compartimento vacío & \textbf{cumple} & demostración simbólica para todo parámetro positivo \\
(A3) & sin nuevas infecciones en no infectados & \textbf{cumple} & por construcción \\
(A4) & el subespacio libre de infección es invariante & \textbf{cumple} & demostración simbólica para todo parámetro positivo \\
(A5) & el DFE es estable sin nuevas i

## 2. Modelos que violan cada supuesto

Cada modelo de esta sección viola **un solo** supuesto; los otros cuatro se siguen cumpliendo.

### (A1) Nuevas infecciones negativas

Si se indica a mano `new_infections={"I": "beta*S*I*(1 - k*I)"}`, ese término es negativo cuando $kI > 1$: no es una tasa de nuevas infecciones. El informe muestra un contraejemplo.

In [6]:
texto_a1 = '''
    dS/dt = Lambda - beta*S*I - mu*S
    dI/dt = beta*S*I*(1 - k*I) - (gamma + mu)*I
'''
a1 = R0Model(texto_a1, infected=["I"], new_infections={"I": "beta*S*I*(1 - k*I)"})
informe_a1 = a1.check_assumptions(language="es")
print(informe_a1.status)
seccion(informe_a1, "A1")

{'A1': 'fails', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A1) flujos no negativos

Si $x \ge 0$, entonces $\mathcal{F}_i,\ \mathcal{V}_i^+,\ \mathcal{V}_i^- \ge 0$ para $i = 1,\dots,n$.

Resultado: **no cumple** (muestras aleatorias de los parámetros).

* $\mathcal{F}_{I} = I S \beta \left(- I k + 1\right) \ge 0$: **no cumple**, muestras aleatorias de los parámetros; negativo para algún $x \ge 0$ en las 40 muestras de parámetros (contraejemplo: $I = 855.4,\ S = 0.8572,\ \beta = 0.06447,\ k = 0.01121$)
* $\mathcal{V}_i^+$ y $\mathcal{V}_i^-$ reúnen los términos negativos y positivos de $\mathcal{V}_i$; son no negativos cuando cada término tiene signo definido.


La descomposición automática separa $\beta S I - \beta k S I^2$ en $\mathcal{F}_I = \beta S I$ y una salida $\beta k S I^2$ en $\mathcal{V}_I^-$, y así se cumplen los cinco supuestos:

In [7]:
print(R0Model(texto_a1, infected=["I"]).check_assumptions().status)

{'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A2) Salidas desde un compartimento vacío

Una cosecha constante $h$ de susceptibles sigue actuando cuando $S = 0$, lo que puede volver negativo a $S$. La librería calcula igual un $\mathcal{R}_0$, pero el modelo no cumple (A2).

In [8]:
a2 = R0Model('''
    dS/dt = Lambda - mu*S - h - beta*S*I
    dI/dt = beta*S*I - (gamma + mu)*I
''', infected=["I"])
print("R0 =", a2.R0)
informe_a2 = a2.check_assumptions(language="es")
print(informe_a2.status)
seccion(informe_a2, "A2")

R0 = -beta*(-Lambda + h)/(mu*(gamma + mu))
{'A1': 'holds', 'A2': 'fails', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A2) sin salidas de un compartimento vacío

Si $x_i = 0$, entonces $\mathcal{V}_i^- = 0$.

Resultado: **no cumple** (demostración simbólica para todo parámetro positivo).

* $\left.\mathcal{V}_{S}^-\right\rvert_{S = 0} = h$: **no cumple** (contraejemplo: $h = 0.01121$)
* $\left.\mathcal{V}_{I}^-\right\rvert_{I = 0} = 0$: **cumple**


### (A3) Por construcción

pyR0compute solo define $\mathcal{F}_i$ en los compartimentos infectados, así que (A3) se cumple siempre. Si se intenta asignar nuevas infecciones a un compartimento no infectado, la librería lo rechaza:

In [9]:
try:
    R0Model('''
        dS/dt = Lambda - beta*S*I - mu*S
        dI/dt = beta*S*I - (gamma + mu)*I
    ''', infected=["I"], new_infections=["beta*S*I", "beta*S*I"])
except Exception as error:
    print(type(error).__name__, ":", error)

ModelSpecificationError : A non-zero new-infection term was given for the uninfected compartment 'S'.


### (A4) Recaída desde un compartimento declarado no infectado

Si los recuperados $R$ recaen a $I$ con tasa $\varepsilon$ (como en tuberculosis o malaria por *P. vivax*), $R$ contiene infección latente. Declararlo no infectado viola (A4): en el subespacio libre de infección ($I = 0$) sigue habiendo entrada a $I$ desde $R$. El Lema 1 también falla, porque $\partial \mathcal{V}_I / \partial R (x_0) = -\varepsilon \neq 0$: la matriz de próxima generación construida solo con $I$ ignora la recaída.

In [10]:
texto_a4 = '''
    dS/dt = Lambda - beta*S*I - mu*S
    dI/dt = beta*S*I - (gamma + mu)*I + eps*R
    dR/dt = gamma*I - (mu + eps)*R
'''
a4 = R0Model(texto_a4, infected=["I"])
informe_a4 = a4.check_assumptions(language="es")
print(informe_a4.status)
seccion(informe_a4, "A4")

{'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'fails', 'A5': 'holds'}


### (A4) el subespacio libre de infección es invariante

Si $x \in X_s$, entonces $\mathcal{F}_i(x) = 0$ y $\mathcal{V}_i^+(x) = 0$ para $i = 1,\dots,m$.

Resultado: **no cumple** (demostración simbólica para todo parámetro positivo).

* $\left.\mathcal{F}_{I}\right\rvert_{X_s} = 0$: **cumple**
* $\left.\mathcal{V}_{I}^+\right\rvert_{X_s} = R eps$: **no cumple** (contraejemplo: $R = 10.84,\ eps = 0.01327$)
* $X_s = \{x \ge 0 : x_i = 0,\ i \le m\}$; los no infectados quedan libres, no fijos en $x_0$.


Al declarar $R$ como infectado se cumplen los cinco supuestos, y $\mathcal{R}_0$ cambia:
$$
\mathcal{R}_0^{(R\ \text{no infectado})} = \frac{\Lambda\beta}{\mu(\gamma+\mu)}
\qquad\text{frente a}\qquad
\mathcal{R}_0^{(R\ \text{infectado})} = \frac{\Lambda\beta\,(\varepsilon+\mu)}{\mu^2(\varepsilon+\gamma+\mu)}.
$$
Con los valores de abajo, ignorar la recaída subestima $\mathcal{R}_0$ en casi tres veces.

In [11]:
a4_bien = R0Model(texto_a4, infected=["I", "R"])
print("Supuestos con I y R infectados:", a4_bien.check_assumptions().status)
print("R0 =", sp.factor(a4_bien.R0))

valores_a4 = dict(Lambda=10, beta=0.02, mu=0.1, gamma=0.5, eps=0.3)
print(f"\nR0 con R no infectado: {a4.R0_numeric(valores_a4):.4f}")
print(f"R0 con R infectado:    {a4_bien.R0_numeric(valores_a4):.4f}")

Supuestos con I y R infectados: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}
R0 = Lambda*beta*(eps + mu)/(mu**2*(eps + gamma + mu))

R0 con R no infectado: 3.3333
R0 con R infectado:    8.8889


### (A5) Un DFE inestable sin infección

Con un efecto Allee fuerte, $\dot S = rS(S/K - 1)$, el equilibrio $S = K$ es inestable aun sin infección. La librería elige por defecto el DFE con más compartimentos distintos de cero ($S = K$), y el informe muestra que no cumple (A5) y que el otro DFE encontrado ($S = 0$) sí lo cumple.

In [12]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # aviso: se encontraron dos DFE
    allee = R0Model('''
        dS/dt = r*S*(S/K - 1) - beta*S*I
        dI/dt = beta*S*I - g*I
    ''', infected=["I"])
informe_allee = allee.check_assumptions(language="es")
print(informe_allee.status)
seccion(informe_allee, "A5")

{'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'fails'}


### (A5) el DFE es estable sin nuevas infecciones

Con $\mathcal{F} = 0$, todos los valores propios de $Df(x_0)$ tienen parte real negativa; equivalentemente, los de $V$ y los de $J_4$ tienen parte real positiva.

Resultado: **no cumple** (demostración simbólica para todo parámetro positivo).

* $V = \left[\begin{matrix}g\end{matrix}\right]$: **cumple**
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}- r\end{matrix}\right]$: **no cumple**
* $V$: $g > 0$
* $J_4$: $- r \le 0$

Otros equilibrios libres de infección encontrados (`dfe_candidates`); se puede elegir uno con `dfe={...}`:

* DFE $S = K$ (el utilizado): $J_4$ **no cumple**
* DFE $S = 0$: $J_4$ **cumple**


### (A5) $V$ no es una M-matriz: la descomposición importa

Si las células infectadas proliferan ($aI$) y ese término se cuenta como transición, $V = g - a$. Cuando $a > g$, $V$ tiene un valor propio negativo y la fórmula da un $\mathcal{R}_0$ **negativo**, sin significado. Sin valores de los parámetros el supuesto **no se pudo decidir**, porque depende de $a$ frente a $g$; con `values=` se decide.

In [13]:
texto_p = '''
    dS/dt = L - mu*S - beta*S*I
    dI/dt = beta*S*I + a*I - g*I
'''
prolif = R0Model(texto_p, infected=["I"])
print("R0 =", prolif.R0)
print("Sin valores:", prolif.check_assumptions().status["A5"])

valores_p = dict(L=1, mu=0.1, beta=0.5, a=0.5, g=0.2)        # a > g
informe_p = prolif.check_assumptions(values=valores_p, language="es")
print("Con a = 0.5 > g = 0.2:", informe_p.status["A5"])
print("Con a = 0.1 < g = 0.2:", prolif.check_assumptions(values=dict(valores_p, a=0.1)).status["A5"])
seccion(informe_p, "A5")

R0 = -L*beta/(mu*(a - g))
Sin valores: undecided
Con a = 0.5 > g = 0.2: fails


Con a = 0.1 < g = 0.2: holds


### (A5) el DFE es estable sin nuevas infecciones

Con $\mathcal{F} = 0$, todos los valores propios de $Df(x_0)$ tienen parte real negativa; equivalentemente, los de $V$ y los de $J_4$ tienen parte real positiva.

Resultado: **no cumple** (en los valores de los parámetros dados).

* $V = \left[\begin{matrix}- a + g\end{matrix}\right]$: **no cumple**, en los valores de los parámetros dados
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}\mu\end{matrix}\right]$: **cumple**
* $V$: $- a + g$
* valores propios de $V$ en los valores dados: $-0.3$
* $J_4$: $\mu > 0$


Si la proliferación se cuenta como nueva infección, `new_infections={"I": "beta*S*I + a*I"}`, entonces $V = g$ y los cinco supuestos se cumplen para todo parámetro. Comparamos ambas descomposiciones con el signo de la mayor parte real de los valores propios del Jacobiano completo en el DFE:

In [14]:
prolif_F = R0Model(texto_p, infected=["I"], new_infections={"I": "beta*S*I + a*I"})
print("Supuestos:", prolif_F.check_assumptions().status)
print("R0 =", sp.simplify(prolif_F.R0), "\n")

f = sp.Matrix([prolif.equations[x] for x in prolif.variables])
J = f.jacobian(list(prolif.variables))
for v in [valores_p, dict(valores_p, a=0.1), dict(valores_p, a=0.1, L=0.01)]:
    sub = {s: v[s.name] for s in prolif.R0.free_symbols}
    r0_v = float(prolif.R0.subs(sub))
    r0_f = float(prolif_F.R0.subs({s: v[s.name] for s in prolif_F.R0.free_symbols}))
    Jn = J.subs(prolif.dfe)
    Jn = Jn.subs({s: v[s.name] for s in Jn.free_symbols})
    abscisa = np.max(np.linalg.eigvals(np.array(Jn.evalf(), dtype=float)).real)
    print(f"a = {v['a']:<4} L = {v['L']:<5} R0 (aI en V) = {r0_v:8.3f}   R0 (aI en F) = {r0_f:7.3f}   máx Re(λ) = {abscisa:+.3f}")

Supuestos: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}
R0 = (L*beta + a*mu)/(g*mu) 

a = 0.5  L = 1     R0 (aI en V) =  -16.667   R0 (aI en F) =  27.500   máx Re(λ) = +5.300
a = 0.1  L = 1     R0 (aI en V) =   50.000   R0 (aI en F) =  25.500   máx Re(λ) = +4.900


a = 0.1  L = 0.01  R0 (aI en V) =    0.500   R0 (aI en F) =   0.750   máx Re(λ) = -0.050


Cuando los supuestos se cumplen (aquí, con $aI$ en $\mathcal{F}$, o con $a < g$ en ambos casos), $\mathcal{R}_0 > 1$ coincide con un DFE inestable y $\mathcal{R}_0 < 1$ con uno estable. Los valores de $\mathcal{R}_0$ difieren entre descomposiciones, pero el umbral es el mismo. Con $a > g$ y $aI$ en $\mathcal{V}$, la fórmula da $\mathcal{R}_0 < 0$ aunque el DFE es inestable.

`R0_numeric` devuelve el radio espectral, que es el valor absoluto, así que oculta el signo. Por eso conviene revisar `check_assumptions()`.

## 3. Modelos grandes: decidir con `values=`

En el modelo de 7 ecuaciones del cuaderno 03, (A1) a (A4) se demuestran simbólicamente. Para (A5), el bloque $\{H, R\}$ de $J_4$ depende de $H^*$ y $R^*$, que son raíces de una cuadrática, y la estabilidad no se puede demostrar para todo parámetro. Con muestras aleatorias el DFE resulta estable en todas, pero eso no es una demostración, así que (A5) queda en **no se pudo decidir**. Con los valores de los parámetros se decide.

In [15]:
inmune = R0Model('''
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
''', infected=["I", "V"])

sin_valores = inmune.check_assumptions(language="es")
print("Sin valores:", sin_valores.status)
print("  (A5):", sin_valores["A5"].items[1].decision.note["es"])

valores = dict(
    g_E=1.0, d_E=0.1, tau_E=2.0, z_VE=1.0, d_I=0.5, tau_I=0.5, z_CI=1.0,
    nu=10.0, d_V=1.0, tau_V=0.5, z_BV=1.0,
    g_H=1.0, d_H=1.0, tau_H=0.5, rho_H=0.3, z_IH=1.0, z_RH=1.0,
    g_C=1.0, d_C=1.0, tau_C=0.5, rho_C=0.3, z_HC=1.0, z_RC=1.0,
    g_B=1.0, d_B=1.0, tau_B=0.5, rho_B=0.3, z_HB=1.0, z_RB=1.0,
    g_R=1.0, d_R=1.0, tau_R=0.5, rho_R=0.3, z_HR=1.0,
)
con_valores = inmune.check_assumptions(values=valores, language="es")
print("\nCon valores:", con_valores.status)
seccion(con_valores, "A5")

Sin valores: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'undecided'}
  (A5): no demostrado; estable en las 51 muestras factibles (use values=... para decidir en un punto)



Con valores: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A5) el DFE es estable sin nuevas infecciones

Con $\mathcal{F} = 0$, todos los valores propios de $Df(x_0)$ tienen parte real negativa; equivalentemente, los de $V$ y los de $J_4$ tienen parte real positiva.

Resultado: **cumple** (en los valores de los parámetros dados).

* $V = \left[\begin{matrix}\frac{C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}}{C^{*} + z_{CI}} & 0\\- \nu & \frac{B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}}{B^{*} + z_{BV}}\end{matrix}\right]$: **cumple**
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}d_{E} & 0 & 0 & 0 & 0\\0 & \frac{R^{*} \rho_{H}}{R^{*} + z_{RH}} + d_{H} & 0 & 0 & H^{*} \left(- \frac{R^{*} \rho_{H}}{\left(R^{*} + z_{RH}\right)^{2}} + \frac{\rho_{H}}{R^{*} + z_{RH}}\right)\\0 & C^{*} \left(\frac{H^{*} \tau_{C}}{\left(H^{*} + z_{HC}\right)^{2}} - \frac{\tau_{C}}{H^{*} + z_{HC}}\right) & - \frac{H^{*} \tau_{C}}{H^{*} + z_{HC}} + \frac{R^{*} \rho_{C}}{R^{*} + z_{RC}} + d_{C} & 0 & C^{*} \left(- \frac{R^{*} \rho_{C}}{\left(R^{*} + z_{RC}\right)^{2}} + \frac{\rho_{C}}{R^{*} + z_{RC}}\right)\\0 & B^{*} \left(\frac{H^{*} \tau_{B}}{\left(H^{*} + z_{HB}\right)^{2}} - \frac{\tau_{B}}{H^{*} + z_{HB}}\right) & 0 & - \frac{H^{*} \tau_{B}}{H^{*} + z_{HB}} + \frac{R^{*} \rho_{B}}{R^{*} + z_{RB}} + d_{B} & B^{*} \left(- \frac{R^{*} \rho_{B}}{\left(R^{*} + z_{RB}\right)^{2}} + \frac{\rho_{B}}{R^{*} + z_{RB}}\right)\\0 & R^{*} \left(\frac{H^{*} \tau_{R}}{\left(H^{*} + z_{HR}\right)^{2}} - \frac{\tau_{R}}{H^{*} + z_{HR}}\right) & 0 & 0 & - \frac{H^{*} \tau_{R}}{H^{*} + z_{HR}} + d_{R} + \rho_{R}\end{matrix}\right]$: **cumple**, en los valores de los parámetros dados
* $V$: $\frac{C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}}{C^{*} + z_{CI}} > 0$, $\frac{B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}}{B^{*} + z_{BV}} > 0$
* $J_4$: $d_{E} > 0$, $\operatorname{tr} = - \frac{H^{*} \tau_{R}}{H^{*} + z_{HR}} + \frac{R^{*} \rho_{H}}{R^{*} + z_{RH}} + d_{H} + d_{R} + \rho_{R},\ \det = \frac{\left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{H} d_{R} + \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{H} \rho_{R} - \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{H} \tau_{R} + \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{R} \rho_{H} + \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} \rho_{H} \rho_{R} - \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} \rho_{H} \tau_{R} + 2 \left(H^{*}\right)^{2} R^{*} d_{H} d_{R} z_{RH} + 2 \left(H^{*}\right)^{2} R^{*} d_{H} \rho_{R} z_{RH} - 2 \left(H^{*}\right)^{2} R^{*} d_{H} \tau_{R} z_{RH} + \left(H^{*}\right)^{2} R^{*} d_{R} \rho_{H} z_{RH} + \left(H^{*}\right)^{2} R^{*} \rho_{H} \rho_{R} z_{RH} - \left(H^{*}\right)^{2} R^{*} \rho_{H} \tau_{R} z_{RH} + \left(H^{*}\right)^{2} d_{H} d_{R} z_{RH}^{2} + \left(H^{*}\right)^{2} d_{H} \rho_{R} z_{RH}^{2} - \left(H^{*}\right)^{2} d_{H} \tau_{R} z_{RH}^{2} + 2 H^{*} \left(R^{*}\right)^{2} d_{H} d_{R} z_{HR} + 2 H^{*} \left(R^{*}\right)^{2} d_{H} \rho_{R} z_{HR} - H^{*} \left(R^{*}\right)^{2} d_{H} \tau_{R} z_{HR} + 2 H^{*} \left(R^{*}\right)^{2} d_{R} \rho_{H} z_{HR} + 2 H^{*} \left(R^{*}\right)^{2} \rho_{H} \rho_{R} z_{HR} - H^{*} \left(R^{*}\right)^{2} \rho_{H} \tau_{R} z_{HR} + 4 H^{*} R^{*} d_{H} d_{R} z_{HR} z_{RH} + 4 H^{*} R^{*} d_{H} \rho_{R} z_{HR} z_{RH} - 2 H^{*} R^{*} d_{H} \tau_{R} z_{HR} z_{RH} + 2 H^{*} R^{*} d_{R} \rho_{H} z_{HR} z_{RH} + 2 H^{*} R^{*} \rho_{H} \rho_{R} z_{HR} z_{RH} + 2 H^{*} d_{H} d_{R} z_{HR} z_{RH}^{2} + 2 H^{*} d_{H} \rho_{R} z_{HR} z_{RH}^{2} - H^{*} d_{H} \tau_{R} z_{HR} z_{RH}^{2} + \left(R^{*}\right)^{2} d_{H} d_{R} z_{HR}^{2} + \left(R^{*}\right)^{2} d_{H} \rho_{R} z_{HR}^{2} + \left(R^{*}\right)^{2} d_{R} \rho_{H} z_{HR}^{2} + \left(R^{*}\right)^{2} \rho_{H} \rho_{R} z_{HR}^{2} + 2 R^{*} d_{H} d_{R} z_{HR}^{2} z_{RH} + 2 R^{*} d_{H} \rho_{R} z_{HR}^{2} z_{RH} + R^{*} d_{R} \rho_{H} z_{HR}^{2} z_{RH} + R^{*} \rho_{H} \rho_{R} z_{HR}^{2} z_{RH} + d_{H} d_{R} z_{HR}^{2} z_{RH}^{2} + d_{H} \rho_{R} z_{HR}^{2} z_{RH}^{2}}{\left(H^{*}\right)^{2} \left(R^{*}\right)^{2} + 2 \left(H^{*}\right)^{2} R^{*} z_{RH} + \left(H^{*}\right)^{2} z_{RH}^{2} + 2 H^{*} \left(R^{*}\right)^{2} z_{HR} + 4 H^{*} R^{*} z_{HR} z_{RH} + 2 H^{*} z_{HR} z_{RH}^{2} + \left(R^{*}\right)^{2} z_{HR}^{2} + 2 R^{*} z_{HR}^{2} z_{RH} + z_{HR}^{2} z_{RH}^{2}}$, $- \frac{H^{*} \tau_{C}}{H^{*} + z_{HC}} + \frac{R^{*} \rho_{C}}{R^{*} + z_{RC}} + d_{C}$, $- \frac{H^{*} \tau_{B}}{H^{*} + z_{HB}} + \frac{R^{*} \rho_{B}}{R^{*} + z_{RB}} + d_{B}$
* valores propios de $J_4$ en los valores dados: $0.9121, 0.9121, 1.106 + 0.08827i, 1.106 - 0.08827i, 0.1$


## Resumen

| Modelo | Supuesto violado | Qué indica el informe |
|---|:--:|---|
| $\mathcal{F}_I = \beta S I (1 - kI)$ dado a mano | (A1) | contraejemplo con $kI > 1$ |
| Cosecha constante $h$ | (A2) | $\mathcal{V}_S^- = h \ne 0$ cuando $S = 0$ |
| Asignar $\mathcal{F}$ a un no infectado | (A3) | la librería lo rechaza |
| Recaída $\varepsilon R \to I$ con $R$ no infectado | (A4) | $\mathcal{V}_I^+ = \varepsilon R$ en $X_s$; declarar $R$ infectado |
| Efecto Allee | (A5) | $J_4 = -r$; el DFE $S = 0$ sí cumple |
| Proliferación $aI$ en $\mathcal{V}$ con $a > g$ | (A5) | $V = g - a < 0$; poner $aI$ en $\mathcal{F}$ |

Si `informe.holds` es `True`, el $\mathcal{R}_0$ calculado es un umbral por el Teorema 2 de van den Driessche y Watmough (2002).